# 04 — Full workflow on your own molecule

Every knob, in the order you meet it, on a molecule that is not one of the
built-ins.

The example is **BeH₂** — small enough to run in under a minute, but a real
choice rather than a bundled one. Substitute your own geometry at the top
and the rest follows.

**Needs:** PySCF. The quantum-solver sections are marked and optional.

If something here disagrees with your expectations, run `quenais-selftest`
first — it checks the install against known-good values in seconds.

### Checklist for a molecule this pipeline has never seen

Work down this list. Each step has a specific failure mode, and skipping one
means finding out about it several hours later.

1. **Geometry in, and written back out.** Use `write_xyz()` to record what was
   *actually* run. A geometry retyped from a paper into a config and never
   echoed back is the cheapest possible source of a wrong answer.
2. **Steps 0–2 first, no solver.** Get the classical references and a valid
   embedding before spending anything on the quantum side.
3. **Check `embedded_scf_check` before reading any other number.** Tolerance
   2e-7 Ha. If it fails, nothing downstream is meaningful.
4. **Look at the active space by hand.** For anything with a d-shell, assume
   ASF under-selects until proven otherwise.
5. **Re-run step 0 with `--force`** once steps 1–2 exist, if you want CASSCF and
   NEVPT2 to use the real active space rather than a first-pass guess.
6. **Run it twice and diff the fingerprints** before treating anything as a
   result.
7. **Only then** consider the quantum solver — and check first whether the
   system can even discriminate selection methods (notebook 05).

### Cost, roughly

| system size | steps 0–2 | notes |
|---|---|---|
| LiH-scale (4–8 qubits) | seconds | run freely |
| N₂-scale (8 qubits) | seconds | |
| ScH-scale (22 qubits) | minutes | GQE training is a separate, much larger cost |
| Cr₂-scale (12e,12o) | ~850k determinants — still exact, but plan for it | |

If ScH-scale steps 0–2 take hours, something is wrong — most likely `eigsh` got
a bad starting vector, or thread count is not set.

## 1. Getting your molecule in

Four ways, tried in this order. Explicit input wins over the built-in
table, so you can override a bundled geometry without renaming anything.

| how | when |
|---|---|
| `geometry="Be 0 0 0; H 0 0 1.33"` | pasting from a paper |
| `xyz="path/to/mol.xyz"` | you have an XYZ file |
| `molecule="LiH"` | one of the built-ins |
| `cif_files/<name>.cif` | crystallographic input |

Units are Angstrom.

In [ ]:
from quenais.utils.geometry import parse_geometry_string, write_xyz

# BeH2 is linear, r(Be-H) ~ 1.33 A.
BEH2 = "Be 0.0 0.0 0.0; H 0.0 0.0 1.33; H 0.0 0.0 -1.33"

geom = parse_geometry_string(BEH2)
for symbol, (x, y, z) in geom:
    print(f"  {symbol:<3} {x:8.4f} {y:8.4f} {z:8.4f}")

# Recording exactly what you ran is worth the one line.
write_xyz(geom, "beh2.xyz", comment="BeH2, linear, r=1.33 A")

## 2. Config: the parameters worth knowing

Molecule identity and paths live on `Config`. Everything else sits in a
settings group, so related knobs stay together and the constructor does
not grow without bound.

Below is every group with the values that matter, and what each one does.

In [ ]:
from quenais import Config
from quenais.settings import (
    AsfSettings, DmetSettings, GqeSettings, QiskitSolverSettings, TierSettings,
)

cfg = Config(
    # ── Identity ─────────────────────────────────────────────────────────
    molecule="BeH2",          # names the cache files; any string
    basis="sto-3g",           # any PySCF basis: 6-31g, cc-pvdz, def2-svp...
    charge=0,
    spin=0,                   # 2S, NOT 2S+1. Only spin=0 is validated.
    geometry=BEH2,            # or xyz="beh2.xyz"
    project_dir="./beh2_run",

    # ── Which classical references to compute (step 0) ───────────────────
    # These are the answer key. CCSD_T is the most trustworthy for
    # main-group systems; CASSCF/NEVPT2 need a sensible active space.
    classical_methods=["HF", "MP2", "CCSD", "CCSD_T"],

    # ── Which quantum solver step 3 uses ─────────────────────────────────
    quantum_solver="sqd",     # sqd | skqd | sqdrift | gqe
)

cfg.validate().make_dirs().load_geometry()
print(cfg)
print("geometry from:", cfg.geometry_source)

### Active-space selection — `cfg.asf`

Which orbitals enter the impurity. The single most consequential choice in
the whole pipeline.

In [ ]:
cfg.asf = AsfSettings(
    # Bypass automatic selection entirely with explicit MO indices
    # (0-based, in the UHF alpha-MO basis).
    #
    # Transition metals NEED this: ASF's entropy thresholds are calibrated
    # on main-group systems and under-select for the d-block. ScH ships
    # with [9, 10, 11, 12, 13, 14] for exactly this reason.
    force_active_space=None,

    # Per-tier entropy thresholds and orbital-count bounds. Tier is chosen
    # automatically: 3 if a transition metal is present, 2 if spin-
    # contaminated or small HOMO-LUMO gap, else 1.
    #   params={1: {"entropy_threshold": 0.05, "max_norb": 12, ...}, ...}

    # Bounds for the gap cutoff that narrows ASF's candidates.
    gap_min_norb=2,
    gap_max_norb=16,

    # Orbitals whose deviation differs by less than this are kept or
    # dropped TOGETHER. Without it, N2's two degenerate pi orbitals get
    # split -- one kept, its partner dropped -- which breaks the
    # molecule's symmetry and gives a physically incomplete space.
    gap_degeneracy_tol=1e-3,

    # Natural-orbital occupation above which an orbital counts as core.
    core_occ_threshold=1.95,

    # Phase C can only ever SHRINK ASF's selection, using a cruder metric
    # than the entropy ASF used to pick them. Set False to keep ASF's
    # full choice.
    phase_c_enabled=True,
)
cfg.asf.validate()
print("active space settings OK")

### The embedding — `cfg.dmet`

How the impurity is coupled to its environment.

In [ ]:
cfg.dmet = DmetSettings(
    # Reference density for the Schmidt decomposition.
    #   "casci" -- CASCI in the ASF space, screened by the core mean field.
    #              Recommended, and the default.
    #   "mp2"   -- reuse step 1's MP2 density. Fast, and unreliable exactly
    #              where static correlation is strong.
    reference="casci",

    # Singular values at or below this are numerical noise, not physics.
    #
    # LOAD-BEARING. On N2 every Schmidt value comes back ~1e-15: the active
    # orbitals are already near-eigenvectors of the reference density, so
    # there is genuinely no bath. Taking the largest values anyway builds
    # physics out of rounding noise -- that produced ~20 Ha of error.
    bath_tolerance=1e-8,

    min_bath_orbs=0,          # zero bath is legitimate; this only warns

    # Ceiling on impurity + bath. Each embedding orbital costs 2 qubits and
    # the embedded CASCI cost grows combinatorially. 18 is the value every
    # reference number was produced with.
    max_embed_orbs=18,

    # Grand-canonical chemical-potential shift. Provably inert for total
    # energies from fixed-particle-number solvers, kept on for solvers
    # that do not fix N.
    mu_correction=True,
    mu_search_range="auto",   # derive the bracket from h1e_emb's spectrum
    mu_max_iter=60,
    mu_tol=1e-10,

    # Flags the embedding when the solver's occupations diverge from the
    # reference density's. Diagnostic only.
    consistency_mismatch_threshold=0.10,
)
cfg.dmet.validate()
print("embedding settings OK")

## 3. Step 0 — classical references

Run these first. Every bug found during development was caught by a
disagreement with a number from this stage.

Watch the `reproducibility` column: CASSCF and NEVPT2 are
optimiser-dependent and will not reproduce to tight tolerance across
machines. The single-determinant methods will.

In [ ]:
from quenais.classical import runner

step0 = runner.main(cfg, force=True)

## 4. Step 1 — active space

For a main-group molecule like BeH₂ the automatic path is fine. For a
transition metal, set `force_active_space` and re-run.

The output tells you which path was taken.

In [ ]:
from quenais.active_space import finder

step1 = finder.main(cfg, force=True)

print()
print(f"tier            : {step1['tier']}")
print(f"active space    : ({step1['nel']}e, {step1['n_active_orbs']}o)")
print(f"MOs             : {step1['mo_list']}")
print(f"forced          : {step1['forced_active_space']}")
print(f"corr strength   : {step1['corr_strength']:.4f}")

### If the active space looks wrong

The diagnostic: a well-chosen active space should put CASSCF+NEVPT2 at or
below CCSD(T). If NEVPT2 lands *above* CCSD(T), the space is too small.

Fix by forcing it:

```python
cfg.asf = AsfSettings(force_active_space=[4, 5, 6, 7])
step1 = finder.main(cfg, force=True)
```

Pick the indices from the deviation spectrum below — look for a clean
break rather than cutting through a cluster of similar values.

**The diagnostic that settles it:** a well-chosen active space puts
CASSCF+NEVPT2 at or *below* CCSD(T). On ScH, NEVPT2 lands **7.2 mHa above**
CCSD(T) (−752.702671 vs −752.709890), which is how the under-selection was
found in the first place.

So: run step 0 with `CASSCF` and `NEVPT2` in `classical_methods`, compare
against `CCSD_T`, and if the multireference method sits above the
single-reference one, your active space is too small — the multireference
treatment has nothing to work with.

The known 3d failure mode is specific: ASF kept 4 orbitals but only **2 active
electrons** on ScH, treating a genuinely valence orbital as core. Every result
in the thesis uses a forced `CAS(4e,6o)` — MOs [9, 10, 11, 12, 13, 14] — instead.

Note MOs 11 and 12 are a degenerate pair with identical entropy. They must be
kept together; the pre-fix `find_gap_cutoff` split pairs like this, which breaks
the molecule's symmetry.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

dev = np.asarray(step1["deviation"])
active = set(step1["mo_list"])
order = np.argsort(-dev)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(range(len(order)),
       dev[order],
       color=["#C44E52" if i in active else "#4C72B0" for i in order])
ax.set_xlabel("orbital (sorted by deviation)")
ax.set_ylabel("min(n, 2-n)")
ax.set_title("red = selected. Look for a clean break, not a cut through a cluster.")

for rank, mo in enumerate(order[:8]):
    print(f"  rank {rank}: MO {mo:3d}  deviation {dev[mo]:.4f}"
          f"{'  <- selected' if mo in active else ''}")

## 5. Step 2 — the embedding

Three things in this output are worth reading rather than skimming.

In [ ]:
from quenais.embedding import hamiltonian

step2 = hamiltonian.main(cfg, force=True)

**The Schmidt singular values** decide the bath. All-zero means no bath,
and that is a correct answer, not a failure.

**The embedded electron count** comes from the reference density, not the
active-space count. They differ whenever there are bath orbitals, and
using the wrong one roughly doubles the energy.

**The embedded SCF check** is the one that can actually fail. It runs a
real converged SCF on the embedding Hamiltonian and compares with
full-molecule UHF. The `ecore` self-consistency identity that some DMET
codes print is tautological — it can never fail — so ignore it.

In [ ]:
print(f"impurity orbitals : {step2['n_imp']}")
print(f"bath orbitals     : {step2['n_bath']}")
print(f"embedding space   : {step2['n_emb']} orbitals ({2*step2['n_emb']} qubits)")
print(f"electrons         : {step2['n_alpha']}a + {step2['n_beta']}b")
print(f"ecore             : {step2['ecore']:.9f} Ha")
print(f"mu                : {step2['mu']:.6f} Ha")
print()
print("Schmidt spectrum  :", np.array2string(np.asarray(step2["sv_all"]),
                                             precision=6, suppress_small=True))
print()
check = step2["embedded_scf_check"]
print(f"embedded SCF      : {check['e_scf_emb']:.9f} Ha")
print(f"full UHF          : {check['e_uhf_full']:.9f} Ha")
print(f"delta             : {check['delta']:+.2e} Ha   within tol: {check['within_tol']}")

### Sanity check against the classical answer

The embedded CASCI is the exact solution *within* the embedding space. It
should land between HF and CCSD(T). If it sits outside that range,
something is wrong upstream — do not proceed to a quantum solver.

In [ ]:
from quenais.visualization.plots import true_embedding_casci

e_hf = step0["methods"]["HF"]["energy"]
e_ccsdt = step0["methods"]["CCSD_T"]["energy"]
e_casci = true_embedding_casci(cfg)

print(f"HF                : {e_hf:.9f} Ha")
print(f"DMET + CASCI      : {e_casci:.9f} Ha")
print(f"CCSD(T)           : {e_ccsdt:.9f} Ha")
print()
recovered = (e_casci - e_hf) / (e_ccsdt - e_hf) * 100
print(f"correlation recovered vs CCSD(T): {recovered:.1f}%")
print(f"sane (between HF and CCSD(T))   : {e_ccsdt <= e_casci <= e_hf}")

## 6. Step 3 — a quantum solver (optional)

### The Qiskit family — `cfg.qiskit`

In-process. Needs `quenais[qiskit]`.

In [ ]:
cfg.quantum_solver = "sqd"        # sqd | skqd | sqdrift

cfg.qiskit = QiskitSolverSettings(
    ansatz="lucj",                # lucj | su2
    fermion_to_qubit="bk",        # bk | jw
    n_shots=8192,
    sqd_iters=10,

    lucj_num_layers=3,
    lucj_random_seed=42,          # pin this if you want reproducible runs
    lucj_regularization=1e-2,

    skqd_krylov_dim=5,            # SKQD only
    skqd_dt=0.9,

    sqdrift_num_circuits=70,      # SqDRIFT only
    sqdrift_time=2.0,

    backend="mps",                # local | mps | ibm
    mps_max_bond_dim=256,
    mps_trunc_thresh=1e-6,
)
cfg.qiskit.validate()
print("qiskit settings OK -- run with: from quenais.quantum import dispatch; dispatch(cfg)")

### The GQE family — `cfg.gqe`

Runs the external CUDA-Q trainer as a subprocess. Needs `quenais[cudaq]`
and a patched `gqe-for-qsci` checkout — see `docs/gqe_integration.md`.

In [ ]:
cfg.gqe = GqeSettings(
    repo_path="../gqe-for-qsci",

    # ── Sampling capacity: the binding constraint on larger systems ──────
    # ScH (22 qubits, ~109k determinants) showed the scaling clearly:
    #   ngates=10, samples=10   -> 60.4 mHa error, stalled at HF
    #   ngates=20, samples=100  -> 36.8 mHa
    #   ngates=40, samples=100  -> 24.1 mHa, subspace hit the cap
    max_iters=120,
    num_samples=100,
    batch_size=100,           # must equal num_samples for online training
    warmup_size=100,
    buffer_size=100,
    ngates=40,                # circuit depth -- often the real limit

    # QSCI subspace cap. The repo default of 2000 became binding on ScH.
    qsci_max_dim=10000,

    # MUST be a dmet_* pool. The stock pools rebuild the molecule from its
    # geometry, and an embedding has none.
    #
    # dmet_excitation accumulates each excitation's Pauli terms into one
    # operator, so it conserves particle number. dmet_pauli_evolution
    # appends them separately and cannot -- roughly half of every sample
    # was discarded as symmetry-violating on ScH.
    operator_pool_spec="dmet_excitation",

    # Backend. qpp-cpu | nvidia | tensornet | tensornet-mps
    cudaq_target="qpp-cpu",

    # True resumes from whatever checkpoint the repo finds, which is not
    # scoped per molecule. Leave False unless resuming deliberately.
    load_checkpoint=False,

    # Including "R-CASCI" triggers a full FCI over the whole embedding
    # space before training starts, purely for logging. Intractable above
    # ~12-16 embedding orbitals.
    reference_keys=None,
)
cfg.gqe.validate()

print("mandatory Hydra overrides that will be sent:")
for override in cfg.gqe.hydra_overrides(cfg.step2_file):
    print("  ", override)

In [ ]:
# Uncomment to actually train (needs cudaq + the patched submodule):
#
# from quenais.quantum import dispatch
# cfg.quantum_solver = "gqe"
# result = dispatch(cfg, force=True)
# print(result)

## 7. Step 4 — figures and summary

Produces whatever the available data supports. With no GQE log, the GQE
figures are skipped rather than drawn empty.

In [ ]:
from quenais.visualization import plots

result = plots.main(cfg)
print()
print(open(result["results_summary"]).read())

## 8. Before you trust the number

Four questions, in order:

1. **Did `quenais-selftest` pass?** If not, nothing else matters.
2. **Is the embedded SCF within 2e-7 Ha of full UHF?** If not, the
   embedding Hamiltonian is wrong regardless of what the energies look
   like.
3. **Is DMET+CASCI between HF and CCSD(T)?** Outside that range means
   something upstream is broken.
4. **For a transition metal: is NEVPT2 at or below CCSD(T)?** If not, the
   active space is too small — force it.

In [ ]:
checks = {
    "embedded SCF vs UHF": abs(step2["embedded_scf_check"]["delta"]) <= 2e-7,
    "CASCI between HF and CCSD(T)": e_ccsdt <= e_casci <= e_hf,
    "bath count is deliberate": step2["n_bath"] >= 0,
    "electron count from density": (step2["n_alpha"], step2["n_beta"]) != (
        step1["nel"] // 2 + step1["nel"] % 2, step1["nel"] // 2
    ) or step2["n_bath"] == 0,
}

for name, ok in checks.items():
    print(f"  [{'ok  ' if ok else 'FAIL'}] {name}")

## 9. Same thing from the command line

```bash
quenais-run --molecule BeH2 --basis sto-3g \
            --geometry "Be 0 0 0; H 0 0 1.33; H 0 0 -1.33" \
            --steps 0 1 2 4 --project-dir ./beh2_run

# from an XYZ file
quenais-run --molecule BeH2 --xyz beh2.xyz --basis sto-3g --steps 0 1 2

# transition metal with a forced active space
quenais-run --molecule ScH --basis sto-3g \
            --force-active-space 9 10 11 12 13 14 --steps 0 1 2
```

## Known limits

Read `docs/limitations.md` before trusting a result from a system unlike
the validated ones. In short: ASF under-selects for transition metals,
CASSCF and NEVPT2 are not tightly reproducible, only closed-shell systems
are validated, and GQE's accuracy on larger systems is bounded by sampling
capacity rather than by anything you can configure away.

---

## Recording what you ran

Reproducing a result six months later needs more than the energy. Keep, per run:

- the **XYZ actually used** (`write_xyz()`, not the config string)
- the **step-2 pickle** — the embedding Hamiltonian everything downstream reads
- the **`--gqe-seed`** if a quantum solver was involved
- the **fingerprints** (`civec_fp`, `order_fp`) if determinant selection was
- the **package version and PySCF build** — deterministic quantities reproduce
  to 1e-10 across installs, but only if you know which install

`tools/compare_pickles.py` diffs two runs key by key with per-quantity
tolerances. It is the tool that catches this project's characteristic failure —
right shape, plausible magnitude, wrong value.

## The standing practice

> **Run every calculation twice. Diff the fingerprints, not just the energy.**

Failure mode #4 in `docs/reproducibility.md` produced **identical total energies
to 14 decimal places from different wavefunctions** — degenerate π orbitals
fixed only up to a rotation, because `symmetry=True` was not set. An
energy-only check would have passed it. The CI-vector fingerprint caught it.

A result that has not been run twice and fingerprinted is not a result yet.